# ACT Model to OpenVINO IR Conversion

The Action Chunking Transformer (ACT) is a model that learns a generative model over action sequences for bimanual manipulation. See the original paper for details: [Action Chunking Transformer](https://arxiv.org/pdf/2304.13705).

In this tutorial, we show how to convert a Unitree ACT policy (stored in the LeRobot format) to the OpenVINO Intermediate Representation (IR), producing FP32 artifacts.


## Dependency and Core Installation Verification
Run the next cell to verify all thre required packages are installed.

In [ ]:
# Dependency Verification & Core Installation (auto-upgrade openvino runtime)
"""
This cell:
  * Verifies core packages (torch, openvino, nncf + utilities)
  * Auto-installs or upgrades openvino runtime to >=2025.0.0 if current version is older
  * Checks for lerobot and EXITS with instructions if it's not present
"""
import sys, subprocess, importlib, pathlib, os
from importlib import metadata

TARGET_OV_VERSION = '2025.0.0'  # Minimum required runtime version
SETUP_SCRIPT = pathlib.Path('setup_unitree_lerobot_env.sh')
README_PATH = pathlib.Path('README.md')

# Core specs excluding openvino (handled separately for upgrade logic)
CORE_SPECS = [
    'openvino>=2025.0.0',
    'nncf>=2.14.0',
    'torch>=2.1', 'torchvision', 'accelerate',
    'safetensors', 'numpy', 'pandas', 'matplotlib', 'tqdm', 'h5py',
    'onnx', 'onnxruntime', 'rich'
]
CORE_IMPORTS = {
    'openvino>=2025.0.0': 'openvino',
    'nncf>=2.14.0': 'nncf',
    'torch>=2.1': 'torch',
    'torchvision': 'torchvision',
    'accelerate': 'accelerate',
    'safetensors': 'safetensors',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'tqdm': 'tqdm',
    'h5py': 'h5py',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'rich': 'rich'
}

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-U'] + list(packages)
    print('[PIP]', ' '.join(cmd))
    subprocess.check_call(cmd)

# --- Upgrade OpenVINO to target version requied for INT8 quantization ---
pip_install(f'openvino>={TARGET_OV_VERSION}')


# --- Remaining core packages ---
print('\n[CHECK] Other core package presence (excluding openvino)...')
missing = []
for spec, name in CORE_IMPORTS.items():
    try:
        importlib.import_module(name)
        print(f'  [OK] {name}')
    except Exception:
        print(f'  [MISSING] {name} (spec: {spec})')
        missing.append(spec)

if missing:
    print('\n[PHASE] Installing missing packages...')
    for spec in missing:
        pip_install(spec)
else:
    print('[INFO] All other core packages already installed.')

print('\n[RECHECK] Core imports after installation:')
still_missing = []
for spec, name in CORE_IMPORTS.items():
    try:
        importlib.import_module(name)
        print(f'  [OK] {name}')
    except Exception:
        still_missing.append(name)
        print(f'  [FAIL] {name} still missing')
if still_missing:
    print('\n[WARN] Remaining missing packages:', still_missing)
    print('Consider restarting the kernel or resolving version conflicts.')

# --- lerobot presence ---
print('\n[CHECK] lerobot availability...')
try:
    import lerobot
    print('[OK] lerobot present.')
except Exception as e:
    print('[ERROR] lerobot not importable:', e)
    print('\nACTION REQUIRED:')
    print(f'  Run setup script: bash {SETUP_SCRIPT}')
    raise SystemExit(1)

# --- Summary ---
from importlib import metadata as md
ov_ver = md.version('openvino') if 'openvino' in sys.modules or metadata.version('openvino') else 'UNKNOWN'
print(f"\n[SUMMARY] Dependencies verified. openvino runtime >= {TARGET_OV_VERSION} (installed: {ov_ver}). Proceed to path configuration cell.")

Next cell configures all the paths.

In [ ]:
# Configuration Parameters (Paths, Precision, Device)
import os, pathlib

CKPT_DIR = pathlib.Path('act_checkpoint') 
NOTEBOOK_DIR = pathlib.Path('.').resolve()
MODEL_DIR = pathlib.Path(os.getenv('ACT_PROJECT_ROOT', NOTEBOOK_DIR))
CHECKPOINT_PATH = pathlib.Path(os.getenv('ACT_CHECKPOINT', str(CKPT_DIR / 'model.safetensors')))
IR_OUTPUT_DIR = pathlib.Path(os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs'))
IR_OUTPUT_DIR.mkdir(exist_ok=True)
DATASET_ROOT = pathlib.Path(os.getenv('ACT_DATASET_ROOT', str(MODEL_DIR / 'dataset')))
STATS_PATH = pathlib.Path(os.getenv('ACT_STATS_PATH', str(CKPT_DIR / 'stats.json')))

PRECISIONS = ['FP32', 'FP16']
TARGET_DEVICE = os.getenv('ACT_TARGET_DEVICE', 'CPU')

print('Notebook directory:', NOTEBOOK_DIR)
print('Relative checkpoint dir:', CKPT_DIR)
print('Resolved checkpoint file path:', CHECKPOINT_PATH)
print('Dataset root:', DATASET_ROOT)
print('Stats path (may not exist yet):', STATS_PATH)
print('Output directory:', IR_OUTPUT_DIR)
print('Target device (OpenVINO):', TARGET_DEVICE)

## Acquire ACT Checkpoint Assets
Download the ACT model artifacts: `model.safetensors`, `config.json`, and `train_config.json` into `act_checkpoint/`

In [ ]:
# Export environment variables
import os, pathlib
CKPT_DIR = pathlib.Path('act_checkpoint')
os.environ['ACT_CHECKPOINT'] = str(CKPT_DIR / 'model.safetensors')
os.environ['ACT_CONFIG_PATH'] = str(CKPT_DIR / 'config.json')
os.environ['ACT_TRAIN_CONFIG_PATH'] = str(CKPT_DIR / 'train_config.json')
stats_path = CKPT_DIR / 'stats.json'
if stats_path.exists():
    os.environ['ACT_STATS_PATH'] = str(stats_path)
print('[INFO] Checkpoint directory (relative):', CKPT_DIR)
print('[INFO] Environment variables:')
for k in ['ACT_CHECKPOINT','ACT_CONFIG_PATH','ACT_TRAIN_CONFIG_PATH','ACT_STATS_PATH']:
    if k in os.environ:
        print('  ', k, '=', os.environ[k])


In [ ]:
# Load Original ACT Model
import os, json, inspect, pathlib, sys, importlib
from safetensors.torch import load_file
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.configs.types import PolicyFeature, FeatureType, NormalizationMode

CHECKPOINT_PATH = pathlib.Path(os.getenv('ACT_CHECKPOINT', 'act_checkpoint/model.safetensors'))
CONFIG_PATH = pathlib.Path(os.getenv('ACT_CONFIG_PATH', str(CHECKPOINT_PATH.parent / 'config.json')))

print('[LOAD] CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('[LOAD] CONFIG_PATH     =', CONFIG_PATH)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint file not found at {CHECKPOINT_PATH}.\n"
        "Ensure you have: (1) placed model.safetensors in act_checkpoint/, or (2) set ACT_CHECKPOINT env var, then re-run this cell."
    )
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"config.json not found at {CONFIG_PATH}.\n"
        "Place config.json next to the checkpoint (act_checkpoint/config.json) or set ACT_CONFIG_PATH."
    )

with open(CONFIG_PATH, 'r') as f:
    cfg_dict = json.load(f)

# Filter config keys to ACTConfig signature
valid_keys = set(inspect.signature(ACTConfig.__init__).parameters.keys()); valid_keys.discard('self')
filtered_cfg = {k: v for k, v in cfg_dict.items() if k in valid_keys}

# Helper wrappers
def wrap_features(feat_dict):
    return {k: PolicyFeature(type=FeatureType(v['type']), shape=tuple(v['shape'])) for k, v in feat_dict.items()}

def wrap_norm_map(norm_map):
    return {FeatureType(k): NormalizationMode(v) for k, v in norm_map.items()}

if 'input_features' in filtered_cfg:
    filtered_cfg['input_features'] = wrap_features(filtered_cfg['input_features'])
if 'output_features' in filtered_cfg:
    filtered_cfg['output_features'] = wrap_features(filtered_cfg['output_features'])
if 'normalization_mapping' in filtered_cfg:
    filtered_cfg['normalization_mapping'] = wrap_norm_map(filtered_cfg['normalization_mapping'])

act_config = ACTConfig(**filtered_cfg)
act_config.use_vae = False
policy = ACTPolicy(act_config)
weights = load_file(str(CHECKPOINT_PATH))
policy.load_state_dict(weights, strict=False)
policy.eval()
print('Loaded ACTPolicy from safetensors. Params:', sum(p.numel() for p in policy.parameters()))

# Extract dimensions
action_dim = filtered_cfg['output_features']['action'].shape[0]
chunk_size = filtered_cfg.get('chunk_size', 100)
# Camera keys
camera_keys = sorted([k for k in cfg_dict['input_features'] if k.startswith('observation.images.')])
print('Detected cameras:', camera_keys)


In [ ]:
# Inspect Model Architecture and Construct Full Dummy Inputs
import torch 

state_dim = policy.config.input_features['observation.state'].shape[0]
chunk_size = chunk_size  # from previous cell
H, W = 480, 640
cams = camera_keys

# Use shapes from config if specified
image_tensors = []
for cam in cams:
    shape = policy.config.input_features[cam].shape  # e.g. [3, H, W]
    img = torch.zeros(1, *shape, dtype=torch.float32)
    image_tensors.append(img)

state = torch.zeros(1, state_dim, dtype=torch.float32)
action_is_pad = torch.zeros(1, chunk_size, dtype=torch.bool)
action_seq = torch.zeros(1, chunk_size, action_dim, dtype=torch.float32)

env_state = None
if 'observation.environment_state' in policy.config.input_features:
    env_dim = policy.config.input_features['observation.environment_state'].shape[0]
    env_state = torch.zeros(1, env_dim, dtype=torch.float32)

print('State shape:', state.shape)
print('Image shapes:', [t.shape for t in image_tensors])
print('Action pad shape:', action_is_pad.shape)
print('Action seq shape:', action_seq.shape)
if env_state is not None:
    print('Environment state shape:', env_state.shape)


In [ ]:
# Prepare Ordered Inputs
# Order: observation.state, each camera image, action_is_pad, action, optional environment_state
ordered_inputs = [state] + image_tensors + [action_is_pad, action_seq] + ([env_state] if env_state is not None else [])
print('Ordered input tensor shapes:', [t.shape for t in ordered_inputs])


In [ ]:
# Export ACT Model to ONNX
import torch, inspect, os, pathlib

# Ensure IR_OUTPUT_DIR is available
try:
    IR_OUTPUT_DIR
except NameError:
    IR_OUTPUT_DIR = pathlib.Path(os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs'))
    print('[ONNX] IR_OUTPUT_DIR was undefined; set to', IR_OUTPUT_DIR)

# Create directory if missing
IR_OUTPUT_DIR.mkdir(exist_ok=True)

# Build ONNXWrapper
class ONNXWrapper(torch.nn.Module):
    def __init__(self, model, camera_keys):
        super().__init__()
        self.model = model
        self.camera_keys = camera_keys
    def forward(self, observation_state, *cam_inputs_and_rest):
        num_cams = len(self.camera_keys)
        cam_inputs = cam_inputs_and_rest[:num_cams]
        action_is_pad_local = cam_inputs_and_rest[num_cams]
        action_local = cam_inputs_and_rest[num_cams + 1]
        observation_environment_state = None
        if len(cam_inputs_and_rest) > num_cams + 2:
            observation_environment_state = cam_inputs_and_rest[num_cams + 2]
        batch = {'observation.state': observation_state}
        for i, cam_key in enumerate(self.camera_keys):
            batch[cam_key] = cam_inputs[i]
        batch['action_is_pad'] = action_is_pad_local
        batch['action'] = action_local
        batch['observation.images'] = list(cam_inputs)
        if observation_environment_state is not None:
            batch['observation.environment_state'] = observation_environment_state
        prediction = self.model.model(batch)
        if isinstance(prediction, tuple):
            prediction = prediction[0]
        return prediction

onnx_path = IR_OUTPUT_DIR / 'model.onnx'
if not onnx_path.exists():
    # Construct dummy args & input names
    dummy_args = [torch.randn_like(state)]
    input_names = ['observation_state']
    for i, cam_key in enumerate(camera_keys):
        cam_tensor = torch.randn_like(image_tensors[i])
        dummy_args.append(cam_tensor)
        input_names.append(f'observation_images_{i}')
    dummy_args.append(torch.zeros_like(action_is_pad))
    dummy_args.append(torch.zeros_like(action_seq))
    input_names += ['action_is_pad', 'action']
    if env_state is not None:
        dummy_args.append(torch.randn_like(env_state))
        input_names.append('observation_environment_state')

    torch.onnx.export(
        ONNXWrapper(policy, camera_keys),
        tuple(dummy_args),
        str(onnx_path),
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=input_names,
        output_names=['output']
    )
    print('ONNX model exported to', onnx_path)
else:
    print('ONNX already exists:', onnx_path)


In [ ]:
# Convert ONNX to OpenVINO IR
import subprocess, shlex, os, pathlib
MO_OUT_DIR = IR_OUTPUT_DIR  # assumes IR_OUTPUT_DIR defined earlier
IR_FP32_XML = MO_OUT_DIR / 'act_model_fp32.xml'
IR_FP32_BIN = MO_OUT_DIR / 'act_model_fp32.bin'

if IR_FP32_XML.exists() and IR_FP32_BIN.exists():
    print('IR already present, skipping MO conversion:', IR_FP32_XML)
else:
    cmd = (
        f"mo --input_model {IR_OUTPUT_DIR / 'model.onnx'} "
        f"--output_dir {MO_OUT_DIR} --model_name act_model_fp32 --compress_to_fp16=False"
    )
    print('Running Model Optimizer:', cmd)
    try:
        subprocess.run(shlex.split(cmd), check=True)
        # Verify artifacts
        if not IR_FP32_XML.exists() or not IR_FP32_BIN.exists():
            raise FileNotFoundError('Expected act_model_fp32.xml/bin not produced. Check MO logs above.')
        print('MO conversion complete. IR files:', IR_FP32_XML, IR_FP32_BIN)
    except Exception as e:
        raise RuntimeError(f'Model Optimizer failed. Ensure openvino (and optionally openvino-dev) installed. Original error: {e}')

print('The following IR files present:')
for f in [IR_FP32_XML, IR_FP32_BIN]:
    print('-', f.name, 'exists' if f.exists() else 'MISSING', '| size:', f.stat().st_size if f.exists() else 0)

### Optional: Direct PyTorch → OpenVINO IR (No ONNX)
Convert the loaded ACT policy directly from PyTorch to OpenVINO IR using the FX frontend. This bypasses ONNX export.

Precision notes:
- Default below is FP32 (weights stored as 32-bit floating point).
- To get FP16 (half-precision weights) via the direct path you can call `ov.save_model(..., compress_to_fp16=True)`.

How to produce FP16 here (two approaches):
Add after FP32 save:
   ```python
   ov.save_model(ov_model, str(IR_OUTPUT_DIR / 'act_model_direct_fp16.xml'), compress_to_fp16=True)
   ```
   This emits `act_model_direct_fp16.xml/bin`.

What gets converted when using `compress_to_fp16=True`?
- Parameter tensors (weights) are stored in FP16.
- Graph topology and layer semantics stay the same.

In [ ]:
# Direct PyTorch to OpenVINO IR (FP32 by default, FP16 instructions included)
import torch, pathlib, openvino as ov

# Required objects from previous cells.
required = ['policy', 'camera_keys', 'state', 'image_tensors', 'action_dim', 'chunk_size']
for sym in required:
    if sym not in globals():
        raise RuntimeError(f'Missing `{sym}`. Run earlier cells first.')

env_present = 'env_state' in globals() and env_state is not None

class DirectWrapper(torch.nn.Module):
    """Expose only observation_state, per-camera images, optional env.
    Internal temporal tensors (action_is_pad, action) are synthesized so they do NOT become IR inputs.
    This keeps the IR minimal and matches evaluation expectations.
    """
    def __init__(self, act_policy, camera_keys, chunk_size, action_dim, env_present=False):
        super().__init__()
        self.model = act_policy
        self.camera_keys = camera_keys
        self.chunk_size = chunk_size
        self.action_dim = action_dim
        self.env_present = env_present
    def forward(self, observation_state, *cams_and_env):
        num_cams = len(self.camera_keys)
        cam_tensors = cams_and_env[:num_cams]
        env_tensor = cams_and_env[num_cams] if self.env_present and len(cams_and_env) > num_cams else None
        B = observation_state.shape[0]
        device = observation_state.device
        action_is_pad_local = torch.zeros(B, self.chunk_size, dtype=torch.bool, device=device)
        action_local = torch.zeros(B, self.chunk_size, self.action_dim, dtype=torch.float32, device=device)
        batch = {
            'observation.state': observation_state,
            'action_is_pad': action_is_pad_local,
            'action': action_local,
            'observation.images': list(cam_tensors)
        }
        for i, key in enumerate(self.camera_keys):
            batch[key] = cam_tensors[i]
        if env_tensor is not None:
            batch['observation.environment_state'] = env_tensor
        out = self.model.model(batch)
        if isinstance(out, tuple):
            out = out[0]
        return out

# Construct example inputs for tracing
example_inputs = [torch.randn_like(state)] + [torch.randn_like(t) for t in image_tensors]
if env_present:
    example_inputs.append(torch.randn_like(env_state))

wrapper = DirectWrapper(policy, camera_keys, chunk_size, action_dim, env_present).eval()
print('[DIRECT] Converting via ov.convert_model (FX)...')
ov_model = ov.convert_model(wrapper, example_input=tuple(example_inputs))

# Rename ports to evaluation expectations
inputs = ov_model.inputs
expected = 1 + len(camera_keys) + (1 if env_present else 0)
if len(inputs) != expected:
    raise RuntimeError(f'Unexpected IR input count {len(inputs)} vs expected {expected}.')

def _set_names(inp, desired: str):
    node = inp.get_node()
    try:
        node.set_friendly_name(desired)
    except Exception:
        pass
    try:
        inp.get_tensor().set_names({desired})
    except Exception as e:
        print('[WARN] Failed to set tensor name for', desired, ':', e)

_set_names(inputs[0], 'observation_state')
for i in range(len(camera_keys)):
    _set_names(inputs[i+1], f'observation_images_{i}')
if env_present:
    _set_names(inputs[-1], 'observation_environment_state')

print('[DIRECT] Final IR input ports (friendly_name / tensor names / partial shape):')
dynamic_present = False
for inp in ov_model.inputs:
    node = inp.get_node()
    try:
        tnames = list(inp.get_tensor().get_names())
    except Exception:
        tnames = []
    try:
        ps = inp.get_partial_shape()
    except Exception:
        ps = None
    if ps is not None and ps.is_static:
        try:
            concrete = ps.to_shape()
            shape_repr = '[' + ', '.join(str(d) for d in concrete) + ']'
        except Exception:
            shape_repr = str(ps)
    else:
        dynamic_present = True
        shape_repr = str(ps) if ps is not None else 'Unknown(dynamic)'
    print(f"  - {tnames[0] if tnames else node.get_friendly_name()} | tensor_names={tnames} | partial_shape={shape_repr}")

IR_OUTPUT_DIR = pathlib.Path(globals().get('IR_OUTPUT_DIR', 'openvino_ir_outputs'))
IR_OUTPUT_DIR.mkdir(exist_ok=True)

# Save FP32 IR
xml_fp32 = IR_OUTPUT_DIR / 'act_model_direct_fp32.xml'
ov.save_model(ov_model, str(xml_fp32))
print('[DIRECT] Saved FP32 XML:', xml_fp32, '| size:', xml_fp32.stat().st_size if xml_fp32.exists() else 0)
print('[DIRECT] Saved FP32 BIN :', xml_fp32.with_suffix('.bin'), '| size:', xml_fp32.with_suffix('.bin').stat().st_size if xml_fp32.with_suffix('.bin').exists() else 0)

# --- FP16 Guidance ---
# To also emit an FP16 version (weights compressed to half precision) uncomment:
# xml_fp16 = IR_OUTPUT_DIR / 'act_model_direct_fp16.xml'
# ov.save_model(ov_model, str(xml_fp16), compress_to_fp16=True)
# print('[DIRECT] Saved FP16 XML:', xml_fp16)
# print('[DIRECT] Saved FP16 BIN :', xml_fp16.with_suffix('.bin'))

print('\n[HINT] In evaluation build input dict using: observation_state, observation_images_0..N, (optional) observation_environment_state.')
print('[DONE] Direct conversion complete. (See comments above for FP16 save).')


### Optional: INT8 Quantization (Post-Training)
This section generates an INT8 (quantized) OpenVINO model using the helper script `quantize_int8_helper.py` found in this folder.

Why INT8?
- Smaller binary size.
- Potential throughput / latency gains (depends on CPU / GPU / VPU).
- Usually minimal accuracy drop if calibration data is representative.

What you need first:
1. A FP32 IR (e.g. `act_model_direct_fp32.xml` created above).
2. `stats.json` from training (already exported earlier or placed into `act_checkpoint/`).
3. A local LeRobot dataset root with episode data (env var `ACT_DATASET_ROOT` or edit path below).
4. Packages: `openvino-dev` and `nncf` installed.

Calibration parameters:
- `num_calib_samples`: how many sequential steps to sample (default 300). Increase if quality degrades.
- `preset`: `performance` (aggressive compression) or `accuracy` (more conservative).

Outputs:
- `int8/model_int8.xml` and `int8/model_int8.bin` in the IR output directory.

Note:
- Typical runtime: ~2–15 minutes for 300 samples on CPU; faster on a modern GPU.

In [ ]:
# Uses quantize_int8_helper.py to produce an INT8 model from the FP32 IR.
# Relies on variables defined earlier: IR_OUTPUT_DIR, STATS_PATH, DATASET_ROOT.
# If the kernel was reset and those are missing, it falls back to env vars or defaults.
# If dataset is missing, prints guidance and shows how to set an override.
import sys, runpy, pathlib, os
from datetime import datetime

IR_OUTPUT_DIR = pathlib.Path(globals().get('IR_OUTPUT_DIR', os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs')))
STATS_JSON = pathlib.Path(globals().get('STATS_PATH', os.getenv('ACT_STATS_PATH', 'act_checkpoint/stats.json')))
DATASET_ROOT = pathlib.Path('dataset/G1_BlockStacking_Dataset')
FP32_XML = IR_OUTPUT_DIR / 'act_model_direct_fp32.xml'
OUT_INT8_DIR = IR_OUTPUT_DIR / 'int8'
CALIB_SAMPLES = 300  # Tune if needed (increase for potentially better accuracy)
PRESET = 'performance'  # or 'accuracy'
SCRIPT_PATH = pathlib.Path('quantize_int8_helper.py')  # Expected in same directory

print('[INT8] Resolved paths:')
print('  IR_OUTPUT_DIR        =', IR_OUTPUT_DIR)
print('  FP32_XML             =', FP32_XML)
print('  STATS_JSON           =', STATS_JSON)
print('  DATASET_ROOT         =', DATASET_ROOT)
print('  SCRIPT_PATH          =', SCRIPT_PATH)
print('  OUT_INT8_DIR         =', OUT_INT8_DIR)
print('  CALIB_SAMPLES        =', CALIB_SAMPLES)
print('  PRESET               =', PRESET)

missing_msgs = []
if not FP32_XML.exists():
    missing_msgs.append(f'FP32 IR not found at {FP32_XML}. Run the direct conversion cell first.')
if not STATS_JSON.exists():
    missing_msgs.append(f'stats.json not found at {STATS_JSON}. Provide training stats for normalization.')
if not DATASET_ROOT.exists():
    missing_msgs.append(f'Dataset root not found at {DATASET_ROOT}. Set ACT_DATASET_ROOT or provide a valid path.')

if not SCRIPT_PATH.exists():
    missing_msgs.append(f'quantize_int8_helper.py not found at {SCRIPT_PATH}. Place the helper script alongside the notebook.')
if missing_msgs:
    raise FileNotFoundError('\n'.join(missing_msgs))

OUT_INT8_DIR.mkdir(exist_ok=True)
print(f'\n[INT8] Starting quantization at {datetime.utcnow().isoformat()}Z')
print(f'[INT8] Using FP32 model: {FP32_XML.name}')
print(f'[INT8] Stats file       : {STATS_JSON.name}')
print(f'[INT8] Dataset root     : {DATASET_ROOT}')
print(f'[INT8] Output directory : {OUT_INT8_DIR}')
print(f'[INT8] Calibration samples={CALIB_SAMPLES} preset={PRESET}')

argv_backup = sys.argv
sys.argv = [
    'quantize_int8_helper.py',
    '--model_xml', str(FP32_XML),
    '--stats_path', str(STATS_JSON),
    '--dataset_root', str(DATASET_ROOT),
    '--output_dir', str(OUT_INT8_DIR),
    '--num_calib_samples', str(CALIB_SAMPLES),
    '--preset', PRESET
]
print('[INT8] Running helper script with args:\n ', ' '.join(sys.argv))
try:
    runpy.run_path(str(SCRIPT_PATH), run_name='__main__')
finally:
    sys.argv = argv_backup

INT8_XML = OUT_INT8_DIR / 'model_int8.xml'
if INT8_XML.exists():
    print('[INT8] Success. INT8 model at', INT8_XML)
    print('[INT8] File sizes: XML', INT8_XML.stat().st_size, 'BIN', INT8_XML.with_suffix('.bin').stat().st_size)
else:
    print('[INT8] Quantization finished but INT8 artifact missing. Check logs above for errors.')


Next cell runs evaluation and comparison for each OpenVINO IR model variant (FP32, MO FP32, INT8) using the helper script. It generates action comparison plots for each variant, comparing OpenVINO outputs to the baseline PyTorch model. Results are saved as PNG figures for further analysis.

In [ ]:
# Evaluation & Comparison Plotting (ENV VAR path & device + precision hints)
"""
Sets OPENVINO_MODEL_PATH, STATS_PATH, OPENVINO_DEVICE, and OPENVINO_PRECISION_HINT per variant.

"""
import sys, pathlib, subprocess, os, datetime, shutil

for sym in ['IR_OUTPUT_DIR', 'CHECKPOINT_PATH', 'STATS_PATH', 'TARGET_DEVICE']:
    if sym not in globals():
        raise RuntimeError(f'Missing required symbol `{sym}`; rerun earlier cells.')

EVAL_SCRIPT = pathlib.Path('eval_openvino_model_helper.py')
if not EVAL_SCRIPT.exists():
    raise FileNotFoundError(f'Helper script missing: {EVAL_SCRIPT}')

# Resolve stats path (fallback to dataset meta)
stats_path = pathlib.Path(STATS_PATH)
DATASET_ROOT = pathlib.Path('dataset/G1_BlockStacking_Dataset')
if not stats_path.exists():
    fallback = DATASET_ROOT / 'meta' / 'stats.json'
    if fallback.exists():
        stats_path = fallback
    else:
        raise FileNotFoundError(f'stats.json not found at {STATS_PATH} or {fallback}')

# Gather variants dynamically (include fp16 if present)
MODEL_VARIANTS = [
    ('direct_fp32', IR_OUTPUT_DIR / 'act_model_direct_fp32.xml'),
    ('direct_fp16', IR_OUTPUT_DIR / 'act_model_direct_fp16.xml'),
    ('mo_fp32', IR_OUTPUT_DIR / 'act_model_fp32.xml'),
    ('int8', IR_OUTPUT_DIR / 'int8' / 'model_int8.xml'),
]
MODEL_VARIANTS = [(lbl, p) for lbl, p in MODEL_VARIANTS if p.exists()]
if not MODEL_VARIANTS:
    raise RuntimeError('No model variants found. Convert / quantize first.')

print('[EVAL] Variants:', ', '.join(lbl for lbl,_ in MODEL_VARIANTS))
print('[EVAL] Stats path:', stats_path)
print('[EVAL] Root path  :', DATASET_ROOT)
print('[EVAL] Policy path:', CHECKPOINT_PATH.parent)
print('[EVAL] Device     :', TARGET_DEVICE)

base_env = os.environ.copy()
base_env['PYTHONWARNINGS'] = 'ignore'
base_env['OPENVINO_DEVICE'] = TARGET_DEVICE
figures = []

def infer_precision(label, path):
    lp = label.lower()
    fname = str(path).lower()
    if 'int8' in lp or 'int8' in fname:
        return 'INT8'
    if 'fp16' in lp or 'fp16' in fname:
        return 'FP16'
    return 'FP32'

for label, model_xml in MODEL_VARIANTS:
    precision_hint = infer_precision(label, model_xml)
    print(f"\n[EVAL] Variant '{label}' -> {model_xml} (device={TARGET_DEVICE}, precision={precision_hint})")
    run_env = base_env.copy()
    run_env['OPENVINO_MODEL_PATH'] = str(model_xml)
    run_env['STATS_PATH'] = str(stats_path)
    run_env['OPENVINO_PRECISION_HINT'] = precision_hint
    cmd = [
        sys.executable,
        str(EVAL_SCRIPT),
        '--repo_id=None',
        f'--root={DATASET_ROOT}',
        f'--policy.path={CHECKPOINT_PATH.parent}',
        '--episodes=0',
    ]
    print('[EVAL] CMD:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True, env=run_env)
    except subprocess.CalledProcessError as e:
        print(f"[EVAL][ERROR] {label} failed: {e}")
        continue

    candidates = [pathlib.Path('actions_comparison.png')]
    fig_src = next((c for c in candidates if c.exists()), None)
    if fig_src:
        timestamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
        fig_dst = pathlib.Path(f'{fig_src.stem}_{label}.png')
        if fig_dst.exists():
            fig_dst = pathlib.Path(f'{fig_src.stem}_{label}_{timestamp}.png')
        shutil.move(str(fig_src), str(fig_dst))
        figures.append(fig_dst)
        print('[EVAL] Saved figure ->', fig_dst)
    else:
        print('[EVAL][WARN] No comparison figure produced for', label)

print('\n[EVAL] Summary:')
for f in figures:
    print('  -', f)
if not figures:
    print('[EVAL] No figures generated.')
print('[EVAL] Done.')